In [ ]:
!pip install pandas

In [ ]:
# ==========================
# BLOQUE 1: INGESTA DE DATOS
# ==========================

#Empezamos por importar pandas y numpy para que nos quede en el resto del entorno

import pandas as pd
import numpy as np
import csv

# Acá cargo mi Dataset. Tuve que usar quoting=csv.QUOTE_NONE (o valor 3) para ignorar comillas problemáticas que causan el error de EOF. Ahora no procesa ninguna comilla como delimitador y las lee literalmente como parte del texto
df_tfmkt= pd.read_csv("dataset_futbol_consolidado.csv", quoting=csv.QUOTE_NONE)

In [ ]:
# Hago el display
print("Dataset: Mercado de transferencias -tfmkt-")
display(df_tfmkt.head())


Dataset: Mercado de transferencias -tfmkt-


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0
3,160668,Antonio Colak,2024-08-30,Parma,0.0,0.0,0.0,3.0,31.0,24,Attack,right,IT1,53312.070,58643.28,2500000.0
4,251302,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,24,Midfield,right,L1,63578.140,82651.58,3500000.0


In [ ]:
# =============================
# BLOQUE 2: RADIOGRAFÍA TÉCNICA
# =============================

# ¿Cuáles son las dimensiones exactas del dataset (filas y columnas)?:

# Obtenemos las dimensiones usando .shape. Inicialmente eran muchos datos, por lo que le hice un acotamiento al dataset sacandole columnas y filtrando registros que no eran del interes del proyecto, tal y como mencione en la primer entrega.
filas_tfmkt, columnas_tfmkt = df_tfmkt.shape
print(f"Dimensiones del dataset: {filas_tfmkt} filas y {columnas_tfmkt} columnas.")

# ¿Qué tipos de datos procesa cada columna? ¿Son todos correctos o hay variables numéricas que se leyeron como texto?

# Para esto uso, como se indicó en las clases, .dtypes y .info.
print("\nTipos de datos con .dtypes")
print(df_tfmkt.dtypes)
print("\nInformación detallada con .info()")
df_tfmkt.info()
# El problema de las comillas me hizo que me tomen todas las columnas como object (text) para este dataset. Lo corrijo en la proxima celda.


Dimensiones del dataset: 5290 filas y 16 columnas.

Tipos de datos con .dtypes
player_id                        int64
player_name                     object
transfer_date                   object
selling_club_name               object
goals_prior_season             float64
assists_prior_season           float64
minutes_played_prior_season    float64
national_team_caps             float64
player_age                     float64
contract_left_months             int64
player_position                 object
foot                            object
buying_club_league              object
destination_gdp_per_capita     float64
buying_club_revenue            float64
transfer_fee_eur               float64
dtype: object

Información detallada con .info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5290 entries, 0 to 5289
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   player_id           

"En una celda Markdown, redacta un breve análisis (mínimo 2 observaciones) sobre anomalías o tendencias que detectes en la estadística descriptiva (ej. ¿Hay valores en 0 que deberían ser nulos? ¿La desviación estándar es muy alta?)"

Primero, Voy a corregir el problema de las comillas residuales tanto de los nombres de las columnas como de los registros, para posteriormente convertirlos a sus tipos de datos óptimos (numéricos y fechas), y poder resolver el tercer apartado.

In [ ]:
# Acá limpio las comillas en los nombres de las columnas como vimos en python nativo con .replace, sacandolas y dejando un espacio vacío ('')
df_tfmkt.columns = df_tfmkt.columns.str.replace('"', '', regex=False)

# Ahora hago lo mismo para limpiar las comillas en los valores de todas las columnas de tipo objeto.
for col in df_tfmkt.columns:
    if df_tfmkt[col].dtype == 'object':
        df_tfmkt[col] = df_tfmkt[col].astype(str).str.replace('"', '', regex=False)

# El regrex=False no estaba en lo que vimos, pero lo agrego para ahorrar tiempo y que busque directamente en las comillas y no en toda la cadena de texto (regex)

# Ahora busco convertir columnas numéricas a flotantes/enteros como siempre tuvo que ser...
columnas_numericas_tfmkt = [
    'player_id', 'goals_prior_season', 'assists_prior_season',
    'minutes_played_prior_season', 'national_team_caps', 'player_age',
    'contract_left_months', 'destination_gdp_per_capita',
    'buying_club_revenue', 'transfer_fee_eur'
]

for col in columnas_numericas_tfmkt:
    df_tfmkt[col] = pd.to_numeric(df_tfmkt[col], errors='coerce')
# Uso errors= coerce para que si encuentra un valor NaN que no se puede convertir, que lo deje como un valor nulo

# Quedo solo convertir la columna de fecha a formato datetime.
df_tfmkt['transfer_date'] = pd.to_datetime(df_tfmkt['transfer_date'], errors='coerce')

# Verificamos que los tipos de datos ahora sean correctos
print("TIPOS DE DATOS CORREGIDOS (df_tfmkt)")
print(df_tfmkt.dtypes)
print("\nVista previa del dataset corregido:")
display(df_tfmkt.head())

TIPOS DE DATOS CORREGIDOS (df_tfmkt)
player_id                               int64
player_name                            object
transfer_date                  datetime64[ns]
selling_club_name                      object
goals_prior_season                    float64
assists_prior_season                  float64
minutes_played_prior_season           float64
national_team_caps                    float64
player_age                            float64
contract_left_months                    int64
player_position                        object
foot                                   object
buying_club_league                     object
destination_gdp_per_capita            float64
buying_club_revenue                   float64
transfer_fee_eur                      float64
dtype: object

Vista previa del dataset corregido:


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0
3,160668,Antonio Colak,2024-08-30,Parma,0.0,0.0,0.0,3.0,31.0,24,Attack,right,IT1,53312.070,58643.28,2500000.0
4,251302,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,24,Midfield,right,L1,63578.140,82651.58,3500000.0


In [ ]:
# ====================================
# BLOQUE 2 (cont): RADIOGRAFÍA TÉCNICA
# ====================================

#Ahora si, voy a intentar hacer un análisis sobre anomalías o tendencias que detecte en la estadística descriptiva (lo tomo como un desafío personal ya que me gusta la econometría):

# Primero calculo las métricas estadísticas claves con .describe (cantidad -count-, media -mean, desvío -std-, min, cuartiles, max) para mis variables numéricas
display(df_tfmkt.describe().T.round(2))
#Agregue el .T que transpone columnas por filas para facilitar lectura y el round(2) que, como vimos en clase, redondea a dos los decimales

# Tambien un breve conteo y análsis del mismo estílo .describe (cantidad  -count-, moda -unique- y frecuencia -freq-) pero ahora sobre las categóricas de texto, por eso el [object]
display(df_tfmkt.describe(include=['object']).T)

# Por ultimo una auditoría de valores nulos (NaN) y valores exactamente en cero (0)
auditoria_calidad = pd.DataFrame({
    'Nulos': df_tfmkt.isnull().sum(),
    '% Nulos': (df_tfmkt.isnull().mean() * 100).round(2),
    'Ceros': (df_tfmkt == 0).sum(),
    '% Ceros': ((df_tfmkt == 0).mean() * 100).round(2)
})
display(auditoria_calidad)

,count,mean,min,25%,50%,75%,max,std
player_id,5290.0,317838.285633,7161.0,149716.0,277635.0,429853.0,1193867.0,214255.610674
transfer_date,5290,2019-12-08 07:47:23.251417856,2012-01-01 00:00:00,2017-07-01 00:00:00,2020-07-01 00:00:00,2022-09-01 00:00:00,2024-09-04 00:00:00,NaN
goals_prior_season,5290.0,2.370321,0.0,0.0,0.0,2.0,50.0,4.842569
assists_prior_season,5290.0,1.593384,0.0,0.0,0.0,2.0,32.0,2.914758
minutes_played_prior_season,5290.0,1196.953875,0.0,0.0,695.5,2381.0,4848.0,1297.704005
national_team_caps,4462.0,27.306141,0.0,4.0,14.0,42.0,233.0,31.527559
player_age,5288.0,23.774981,16.4,21.4,23.4,25.7,36.6,3.3167
contract_left_months,5290.0,24.0,24.0,24.0,24.0,24.0,24.0,0.0
destination_gdp_per_capita,5290.0,53088.858136,40409.99,49039.57,53312.07,54856.867,63578.14,5525.465741
buying_club_revenue,5290.0,70983.470486,50610.63,55361.45,60687.22,87581.1,98649.8,17702.955644


,count,unique,top,freq
player_name,5290,3505,Gianluca Caprari,7
selling_club_name,5290,946,Juventus,50
player_position,5290,4,Attack,1694
foot,5290,4,right,3766
buying_club_league,5290,5,GB1,1404


,Nulos,% Nulos,Ceros,% Ceros
player_id,0,0.00,0,0.00
player_name,0,0.00,0,0.00
transfer_date,0,0.00,0,0.00
selling_club_name,0,0.00,0,0.00
goals_prior_season,0,0.00,3124,59.05
assists_prior_season,0,0.00,3168,59.89
minutes_played_prior_season,0,0.00,2183,41.27
national_team_caps,828,15.65,232,4.39
player_age,2,0.04,0,0.00
contract_left_months,0,0.00,0,0.00


Veo un dato dentro de la estadística descriptiva alarmante:
En el tiempo restante de contrato no hay desvío estadístico, y esto es porque todas las transferencias que registra mi dataset fueron, supuestamente, con 24 meses de antelación a la finalización del vínculo. Esto es practicamente imposible, así que cuenta como punto a corregir.

Luego hay datos que a priori pueden parecer raros pero que chequeados le encontré sentido. Ejemplo, Gianluca Camprari, el jugador con mayor numero de transferencias en el período (siete), efectivamente cambio varias veces de club.

En cuanto a la mini-auditoría:
Hay 2 registros nulos en player_age y 6 en foot, lo cual se podría intentar corregir pero, para mi fortuna, son numeros bajos en comparación a la totalidad del dataset (+5000 registros)

Acá tambien hay datos que a priori parecen problematicos como los 828 registros nulos en national_team_caps, pero la explicación que yo le doy a esto es que si no hay registro de que los jugadores hayan participado la temporada anterior en la selección es porque, quizas, no fueron convocados o jugaban para selecciónes de menor nivel, lo que derivó en que nadie les haya llevado la cuenta y al no haber dato certero, directamente no hay ningún numero. Aca quedo expectante a una opinión agena para ver como proceder de cara a las proximas actividades.

In [ ]:
# =======================================================
# BLOQUE 3: LOGICA ALGORITMICA Y FUNCIONES PERSONALIZADAS
# =======================================================

# Para la primer consigna de crear la función, voy a clasificar al futbolista según su ciclo de vida económico al momento del pase.
def categorizar_etapa_carrera(edad):
    if pd.isna(edad):
        return 'Sin Dato'
    elif edad < 21:
        return 'Promesa (<21)'
    elif edad <= 28:
        return 'Prime (21-28)'
    else:
        return 'Veterano (>28)'
# Ahora la función propia la voy a agregar al dataset como una columna más. Para ello uso, tal y como sugieren, el .apply, creando una nueva columna categórica "careear_stage"
df_tfmkt['career_stage'] = df_tfmkt['player_age'].apply(categorizar_etapa_carrera)
# Imprimo un conteo del resultado solo para verificar y por curiosidad
print(df_tfmkt['career_stage'].value_counts())

# Para la segunda consigna de implementar lamda para realizar una transformación matemática sobre una columna, voy convertir las transferencia de euros a millones de euros (redondeado a 2 decimales) y la voy a mostrar en una nueva columna "transfer_fee_millions"
df_tfmkt['transfer_fee_millions'] = df_tfmkt['transfer_fee_eur'].apply(lambda x: round(x / 1_000_000, 2))

# Por medio de este display podemos ver las dos nuevas columnas (al final de todo) en acción.
display(df_tfmkt.head())


career_stage
Prime (21-28)     3631
Promesa (<21)     1079
Veterano (>28)     578
Sin Dato             2
Name: count, dtype: int64


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur,career_stage,transfer_fee_millions
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0,Veterano (>28),4.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0,Veterano (>28),0.5
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0,Veterano (>28),11.9
3,160668,Antonio Colak,2024-08-30,Parma,0.0,0.0,0.0,3.0,31.0,24,Attack,right,IT1,53312.070,58643.28,2500000.0,Veterano (>28),2.5
4,251302,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,24,Midfield,right,L1,63578.140,82651.58,3500000.0,Prime (21-28),3.5
